### Імпорт бібліотек, встановлення налаштувань відображення, завантаження файлу

In [88]:
import pandas as pd
import numpy as np
import re

In [89]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [90]:
df = pd.read_csv("D:\\my_projects\\kursah\\data\\raw\\result_test_100.csv")

In [91]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   price_UAH           100 non-null    str  
 1   price_USD           100 non-null    str  
 2   title               100 non-null    str  
 3   location_list       100 non-null    str  
 4   details_list        100 non-null    str  
 5   advantages_list     100 non-null    str  
 6   description         100 non-null    str  
 7   created_at          100 non-null    str  
 8   facilities_list     100 non-null    str  
 9   technically_tested  100 non-null    str  
 10  seller_name         100 non-null    str  
 11  seller_trust        100 non-null    str  
 12  city                100 non-null    str  
 13  plus_text           100 non-null    str  
 14  url                 100 non-null    str  
 15  page                100 non-null    int64
dtypes: int64(1), str(15)
memory usage: 12.6 KB


## Очистка і формування стовпчиків для аналізу

### Загальні стовпці з ціною, id, кількістю кімнат

In [92]:
df['id'] = df['title'].str.extract(r'ID (\d+)').astype(int)
df['rooms'] = df['title'].str.extract(r'(\d+)к квартири').astype(int)
df['price_UAH'] = df['price_UAH'].str.replace(' ', '').str.extract(r'(\d+)').astype(int)
df['price_USD'] = df['price_USD'].str.replace(' ', '').str.extract(r'(\d+)').astype(int)

### Стовпець дати

In [93]:
months = {
    'січ': '01', 'лют': '02', 'бер': '03', 'кві': '04',
    'тра': '05', 'чер': '06', 'лип': '07', 'сер': '08',
    'вер': '09', 'жов': '10', 'лис': '11', 'гру': '12'
}
extracted = df['created_at'].str.extract(r'(\d+)\s+([а-яіїєґ]+)\.*\s*(\d{4})?')
extracted.columns = ['day', 'month_abbr', 'year']
current_year = str(pd.Timestamp.now().year)
extracted['year'] = extracted['year'].fillna(current_year)
extracted['month'] = extracted['month_abbr'].map(months)

df['date'] = pd.to_datetime(
    extracted['day'] + '.' + extracted['month'] + '.' + extracted['year'],
    format='%d.%m.%Y'
)

### Створення стовпців з локацією

In [94]:
df['city_name'] = df['city'].str.split(',').str[2].str.strip().astype(str)
df['district'] = df['location_list'].str.extract(r'район\s+([^|]+)')[0].str.strip()
df['metro'] = df['location_list'].str.extract(r'метро\s+([^|]+)')[0].str.strip()
df['address'] = df['title'].str.extract(r'на\s+((?:вул|просп|бул|пров|пл|шосе|наб)\.?\s+.+?)(?=\s*•)')[0].str.strip()

### Створення стовпців з інформацією про продавця

In [95]:
df['seller'] = df['seller_name'].str.extract(r'\n([А-ЯІЇЄҐа-яA-Za-z][^\n\d]+?)(?:\n|$)')[0].str.strip()

def experience_to_months(text):
    if pd.isna(text):
        return 0
    
    years  = re.search(r'(\d+)\+?\s*рок|(\d+)\+?\s*рік', text)
    months = re.search(r'(\d+)\s*місяц', text)
    days   = re.search(r'(\d+)\s*дн', text)
    
    total = 0
    if years:
        total += int(years.group(1) or years.group(2)) * 12
    if months:
        total += int(months.group(1))
    if days:
        total += int(days.group(1)) / 30
    
    return total

def categorize(months):
    if months < 12:
        return 'Новачок'        # < 1 року
    elif months < 36:
        return 'Досвідчений'     # 1–3 роки
    elif months < 60:
        return 'Професіонал'  # 3–5 років
    else:
        return 'Експерт'      # 5+ років
    
df['experience_months'] = df['seller_trust'].apply(experience_to_months).round().astype(int)
df['seller_level'] = df['experience_months'].apply(categorize)

print(df[['experience_months', 'seller_level']])

    experience_months seller_level
0                  24  Досвідчений
1                  60      Експерт
2                  60      Експерт
3                  60      Експерт
4                  60      Експерт
5                  60      Експерт
6                  60      Експерт
7                  33  Досвідчений
8                  60      Експерт
9                  25  Досвідчений
10                 60      Експерт
11                 60      Експерт
12                 60      Експерт
13                 60      Експерт
14                 60      Експерт
15                 60      Експерт
16                  7      Новачок
17                  7      Новачок
18                  1      Новачок
19                 60      Експерт
20                 60      Експерт
21                 60      Експерт
22                 60      Експерт
23                 46  Професіонал
24                  0      Новачок
25                 60      Експерт
26                 60      Експерт
27                 6

### Створення стовпців по площі 

In [96]:
df['area_total']   = df['details_list'].str.extract(r'[Зз]агальна площа\s+([\d.]+)')[0]
df['area_living']  = df['details_list'].str.extract(r'житлова\s+([\d.]+)')[0]
df['area_kitchen'] = df['details_list'].str.extract(r'кухня\s+([\d.]+)')[0]

for col in ['area_total', 'area_living', 'area_kitchen']:
    df[col] = df[col].astype(float)

print(df[['area_total', 'area_living', 'area_kitchen']])

    area_total  area_living  area_kitchen
0         74.0          NaN           8.6
1         27.0         16.0           6.0
2        140.0         55.4          34.2
3         70.0          NaN           NaN
4         87.0         60.0          20.0
5         76.0         40.0          15.0
6        129.2         72.3           9.1
7        125.2         66.7          13.6
8         28.0          NaN           NaN
9         50.0          NaN          10.0
10        77.0         30.0          30.0
11        70.0          NaN           NaN
12        45.0          NaN          12.0
13        49.4         15.9          10.1
14        46.0          NaN           NaN
15        41.4         16.3          12.0
16        28.0         14.0           8.0
17        45.0         28.0           6.0
18        46.0         19.6          10.7
19        46.0          NaN           NaN
20        46.0          NaN           NaN
21        50.0         17.0          11.0
22        70.0          NaN       

### Створення стовпців поверху квартири та поверховості будинку

In [97]:
df['floor_current'] = df['technically_tested'].str.extract(r'(\d+)\s+поверх з')[0].astype(int)
df['floor_total']   = df['technically_tested'].str.extract(r'поверх з\s+(\d+)')[0].astype(int)

print(df[['floor_current', 'floor_total']])

    floor_current  floor_total
0               4            5
1               3            4
2               7            8
3              12           27
4               5           16
5              10           23
6              16           17
7              11           12
8              17           28
9               5           20
10              1            7
11              1            9
12              5            6
13              5            5
14              9           10
15              5            8
16              6           25
17              1            5
18             25           34
19              4           15
20              4           15
21              9           11
22              7            8
23              4            4
24              9           23
25             13           14
26              5            9
27             11           15
28              3           18
29              1            9
30              8            9
31      

### Створення бінарних стовпців по особливостям планування

In [98]:
df['kitchen_studio'] = df['plus_text'].str.extract(r'(Кухня-студія)', flags=re.IGNORECASE)
df['penthouse']      = df['plus_text'].str.extract(r'(Пентхаус)', flags=re.IGNORECASE)
df['multilevel']     = df['plus_text'].str.extract(r'(Багаторівнева)', flags=re.IGNORECASE)

for col in ['kitchen_studio', 'penthouse', 'multilevel']:
    df[col] = df[col].notna().astype(int)

print(df[['kitchen_studio', 'penthouse', 'multilevel']])

    kitchen_studio  penthouse  multilevel
0                0          0           0
1                0          0           0
2                1          1           1
3                0          0           0
4                1          0           0
5                1          0           0
6                1          1           0
7                0          0           0
8                1          0           0
9                0          0           0
10               0          0           0
11               0          0           0
12               0          0           0
13               0          0           0
14               0          0           0
15               0          0           0
16               1          0           0
17               0          0           0
18               0          0           0
19               0          0           0
20               0          0           0
21               0          0           0
22               1          0     

### Створення стовпця про наявність укриття 

In [99]:
df['shelter'] = df['advantages_list'].str.extract(r'((?:У|у)криття[^|]+)')[0].str.strip()

def extract_shelter_context(text):
    if pd.isna(text):
        return None
    match = re.search(r'(\S+\s+){0,2}\S*[Уу]крит\S*(\s+\S+){0,2}', text)
    return match.group().strip() if match else None

df['shelter_context'] = df['description'].apply(extract_shelter_context)
df['shelter_plus'] = df['plus_text'].apply(extract_shelter_context)

print(df['shelter'])

0                   NaN
1     Укриття в будинку
2                   NaN
3                   NaN
4                   NaN
5                   NaN
6     Укриття в будинку
7                   NaN
8                   NaN
9     Укриття в будинку
10                  NaN
11                  NaN
12                  NaN
13                  NaN
14                  NaN
15                  NaN
16                  NaN
17                  NaN
18                  NaN
19                  NaN
20                  NaN
21                  NaN
22                  NaN
23                  NaN
24                  NaN
25    Укриття в будинку
26                  NaN
27                  NaN
28                  NaN
29    Укриття в будинку
30                  NaN
31    Укриття в будинку
32                  NaN
33                  NaN
34                  NaN
35                  NaN
36                  NaN
37                  NaN
38                  NaN
39                  NaN
40    Укриття в будинку
41              

Зведення до єдиної бінарної колонки по наявності укриття:

In [100]:
df['shelter_plus'] = df['shelter_plus'].str.contains('укриття', case=False, na=False).astype(int)
df['shelter_context'] = df['shelter_context'].str.contains('укриття', case=False, na=False).astype(int)
df['shelter'] = df['shelter'].str.contains('укриття', case=False, na=False).astype(int)

shelter_columns = ['shelter_plus', 'shelter_context', 'shelter']
df['has_shelter'] = df[shelter_columns].any(axis=1).astype(int)


# print(df[['shelter_plus', 'shelter_context', 'shelter']])
print(df['has_shelter'].value_counts())

has_shelter
0    69
1    31
Name: count, dtype: int64


### Створення стовпця дозволу проживання з тваринами

In [101]:
df['pets'] = df['facilities_list'].str.extract(r'((?:[Мм]ожна|[Нн]е можна|[Бб]ез)\s+з?\s*тварин[^|]*)')[0].str.strip()

def extract_bober_context(text):
    if pd.isna(text):
        return None
    match = re.search(r'(\S+\s+){0,2}\S*[Тт]варин\S*(\s+\S+){0,2}', text)
    return match.group().strip() if match else None

df['pets_desc'] = df['description'].apply(extract_bober_context)

Зведення до єдиної бінарної колонки по дозволу проживання з тваринами:

In [102]:
df['pets'] = df['pets'].fillna('Не вказано')

def classify_pets_desc(text):
    if pd.isna(text):
        return None
    text_lower = text.lower()
    
    if re.search(r'без тварин|не можна|заборон|неможлив|не приймаємо тварин|не розглядаєть|не розглядаємо', text_lower):
        return 'не можна'
    
    if re.search(r'тварин|тваринк', text_lower):
        return 'можна'
    
    return None

df['pets_classified'] = df['pets_desc'].apply(classify_pets_desc)

df['pets'] = df['pets'].fillna(df['pets_classified']).fillna('не вказано')


print(df['pets'].value_counts())

pets
Не вказано           48
Без тварин           39
Можна з тваринами    13
Name: count, dtype: int64


### Створення стовпців забезпеченості при відсутності електроенергії.   

In [103]:
features = {
    'no_light_internet':    'є інтернет',
    'no_light_mobile_connection':      'є мобільний зв\'язок',
    'no_light_water':       'є водопостачання',
    'no_light_heating':     'працює опалення',
    'no_light_gas':         'є газ',
    'no_light_elevator':    'працює ліфт'
}

for col, keyword in features.items():
    df[col] = df['plus_text'].str.contains(keyword, case=False, na=False).astype(int)

print(df[['no_light_internet', 'no_light_mobile_connection', 'no_light_water', 'no_light_heating', 'no_light_gas',
           'no_light_elevator']])

    no_light_internet  no_light_mobile_connection  no_light_water  \
0                   0                           1               1   
1                   1                           0               0   
2                   1                           0               1   
3                   1                           1               1   
4                   1                           0               1   
5                   1                           1               1   
6                   1                           1               1   
7                   1                           0               1   
8                   1                           1               1   
9                   1                           1               1   
10                  0                           0               0   
11                  0                           0               0   
12                  0                           0               0   
13                  0             

### **Створення кінцевого датасету**

In [104]:
cl_df = df[['id', 'rooms', 'price_UAH', 'price_USD', 'date', 'city_name', 'district', 'metro', 'address', 'seller',
            'experience_months', 'seller_level', 'area_total', 'area_living', 'area_kitchen', 'floor_current', 
            'floor_total', 'kitchen_studio', 'penthouse', 'multilevel', 'has_shelter', 'pets', 'no_light_internet',
            'no_light_mobile_connection', 'no_light_water', 'no_light_heating', 'no_light_gas', 'no_light_elevator',
            'url']].copy()

print(cl_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id                          100 non-null    int64         
 1   rooms                       100 non-null    int64         
 2   price_UAH                   100 non-null    int64         
 3   price_USD                   100 non-null    int64         
 4   date                        100 non-null    datetime64[us]
 5   city_name                   100 non-null    str           
 6   district                    97 non-null     str           
 7   metro                       39 non-null     str           
 8   address                     100 non-null    str           
 9   seller                      100 non-null    str           
 10  experience_months           100 non-null    int64         
 11  seller_level                100 non-null    str           
 12  area_t

In [105]:
print(cl_df.head(10))

         id  rooms  price_UAH  price_USD       date city_name  \
0  33854321      3      22175        500 2026-01-18      Київ   
1  33683279      1       8000        180 2025-11-11      Київ   
2  33543548      3      85000       1917 2025-09-29      Київ   
3  33938212      1      53220       1200 2026-02-08      Київ   
4  31091475      3      48785       1100 2026-02-08      Київ   
5  33845795      2      35480        800 2026-03-13     Одеса   
6  34025534      5      51000       1150 2026-03-04      Київ   
7  33580264      3      26610        600 2026-02-01     Одеса   
8  33862733      1      19071        430 2026-01-07      Київ   
9  33604961      1      32000        722 2025-10-17      Київ   

             district              metro                        address  \
0               Нивки       Берестейська         вул. Марка Безручка 29   
1          Дарницький     Червоний хутір   вул. Бориспільська 9, кв. 70   
2               Поділ  Контрактова площа        вул. Дегтяр